# Olist — Kiểm kê và kiểm tra chất lượng dữ liệu

Chạy các cell từ trên xuống trong VS Code bằng kernel Python có `pandas`.
Notebook và thư mục `data/raw` đặt cùng cấp; `data/raw` chứa đúng 9 CSV, mỗi bảng một phiên bản.

**Phạm vi hiện tại:** bước 1 chuẩn bị, bước 2 đọc dữ liệu, bước 3 kiểm tra chất lượng.
Các cell 3.1–3.14 là các mục nhỏ của bước 3; chưa triển khai bước 4 làm sạch.
Chuyển kiểu số/ngày chỉ thực hiện trên biến tạm phục vụ kiểm tra.

Chú thích `# Phát hiện` ghi lại kết quả trên bộ CSV đã kiểm tra. Nếu thay dữ liệu, đọc kết quả chạy mới và cập nhật chú thích.


In [1]:
# Cell 1 — Thư viện và đường dẫn
# Mục đích: Khai báo thư viện và kiểm tra thư mục data/raw trước khi đọc CSV.
# Phát hiện: Đường dẫn tương đối được tính từ Path.cwd(); mỗi bảng chỉ cần một phiên bản CSV.

from pathlib import Path
import re
import pandas as pd

DATA_DIR = Path("data/raw")

print("Thư mục notebook:", Path.cwd())
print("Thư mục dữ liệu:", DATA_DIR.resolve())
print("Các file CSV:", [p.name for p in DATA_DIR.glob("*.csv")])


Thư mục notebook: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11
Thư mục dữ liệu: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11\data\raw
Các file CSV: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:
# Cell 2 — Đọc 9 bảng dữ liệu gốc
# Mục đích: Đọc CSV vào data, giữ ID và mã bưu chính ở dạng string; nhận cả tên có hậu tố (n).
# Phát hiện: Bộ dữ liệu gồm 9 bảng. Nếu thiếu bảng hoặc có nhiều phiên bản, dừng để chọn lại nguồn.

expected = {
    "customers": "olist_customers_dataset",
    "geolocation": "olist_geolocation_dataset",
    "items": "olist_order_items_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "orders": "olist_orders_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "translation": "product_category_name_translation",
}

data = {}

for table, expected_stem in expected.items():
    matches = []

    for path in DATA_DIR.glob("*.csv"):
        stem = re.sub(r"\s*\(\d+\)$", "", path.stem)
        if stem == expected_stem:
            matches.append(path)

    if len(matches) == 0:
        raise FileNotFoundError(f"Không tìm thấy file cho bảng: {table}")

    if len(matches) > 1:
        raise ValueError(
            f"Có nhiều file cho bảng {table}: {[p.name for p in matches]}. "
            "Hãy chỉ giữ một phiên bản trong data/raw."
        )

    data[table] = pd.read_csv(matches[0], dtype="string")
    print(f"{table}: {matches[0].name} — {len(data[table]):,} dòng")


customers: olist_customers_dataset.csv — 99,441 dòng
geolocation: olist_geolocation_dataset.csv — 1,000,163 dòng
items: olist_order_items_dataset.csv — 112,650 dòng
payments: olist_order_payments_dataset.csv — 103,886 dòng
reviews: olist_order_reviews_dataset.csv — 99,224 dòng
orders: olist_orders_dataset.csv — 99,441 dòng
products: olist_products_dataset.csv — 32,951 dòng
sellers: olist_sellers_dataset.csv — 3,095 dòng
translation: product_category_name_translation.csv — 71 dòng


In [3]:
# Cell 3.1 — Tổng quan missing và dòng trùng hoàn toàn
# Mục đích: Thống kê số dòng, số cột, tổng ô thiếu và số dòng trùng hoàn toàn của từng bảng.
# Phát hiện: Reviews, orders và products có missing nhưng không có dòng trùng hoàn toàn.
# Geolocation có 261.831 dòng trùng tính dư sau lần xuất hiện đầu.
# missing_cells đếm ô thiếu; exact_duplicate_rows đếm dòng trùng dư.

quality = pd.DataFrame([
    {
        "table": table,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
    }
    for table, df in data.items()
])

quality


,table,rows,columns,missing_cells,exact_duplicate_rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,translation,71,2,0,0


In [4]:
# Cell 3.2 — Kiểm tra lại dòng trùng và khóa kết hợp của reviews
# Mục đích: Phân biệt trùng toàn bộ dòng với lặp cặp review_id + order_id.
# Phát hiện: Reviews, orders và products đều có 0 dòng trùng hoàn toàn; cặp review_id + order_id lặp 0 dòng.
# Kết quả này không khẳng định review_id đứng riêng là duy nhất.

for table in ["reviews", "orders", "products"]:
    print(table, "dòng trùng hoàn toàn:", data[table].duplicated().sum())

print(
    "Reviews trùng theo review_id + order_id:",
    data["reviews"].duplicated(["review_id", "order_id"]).sum()
)


reviews dòng trùng hoàn toàn: 0
orders dòng trùng hoàn toàn: 0
products dòng trùng hoàn toàn: 0
Reviews trùng theo review_id + order_id: 0


In [5]:
# Cell 3.3 — Missing theo từng cột
# Mục đích: Xác định số lượng và tỷ lệ missing ở reviews, orders và products.
# Phát hiện: Reviews thiếu title 87.656 (88,34%), message 58.247 (58,70%).
# Orders thiếu ngày duyệt 160, ngày bàn giao vận chuyển 1.783, ngày giao khách 2.965.
# Products: 610 missing ở mỗi cột danh mục/độ dài tên/độ dài mô tả/số ảnh;
# 2 missing ở mỗi cột cân nặng và kích thước. Tỷ lệ hiển thị được làm tròn.

for table in ["reviews", "orders", "products"]:
    df = data[table]

    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    report = report[report["missing_count"] > 0]

    print(f"\nBảng: {table}")
    display(report)



Bảng: reviews


,missing_count,missing_percent
review_comment_title,87656,88.34
review_comment_message,58247,58.70



Bảng: orders


,missing_count,missing_percent
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98



Bảng: products


,missing_count,missing_percent
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [7]:
# Cell 3.4 — Missing ngày theo trạng thái đơn hàng
# Mục đích: Phân biệt ngày thiếu phù hợp với trạng thái và các trường hợp cần xem thêm.
# Phát hiện: Delivered có 14 đơn thiếu ngày duyệt, 2 thiếu ngày bàn giao vận chuyển, 8 thiếu ngày giao khách.
# Các nhóm có thể chồng nhau. Có 6 đơn canceled vẫn có ngày giao khách.
# Đơn chưa hoàn thành có thể thiếu ngày giao; chưa điền ngày hoặc xóa đơn.

orders = data["orders"]

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

missing_by_status = (
    orders[date_columns]
    .isna()
    .groupby(orders["order_status"])
    .sum()
)

missing_by_status.insert(
    0,
    "total_orders",
    orders.groupby("order_status").size()
)

display(missing_by_status)


,total_orders,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [8]:
# Cell 3.5 — Kiểm tra bộ khóa dự kiến
# Mục đích: Kiểm tra thiếu thành phần khóa và các dòng thuộc nhóm khóa lặp ở 8 bảng.
# Phát hiện: Cả 8 bộ khóa dự kiến đều thiếu 0 dòng và lặp 0 dòng.
# Reviews dùng review_id + order_id; geolocation được kiểm tra riêng.
# duplicate_key_rows đếm toàn bộ dòng trong nhóm khóa lặp (keep=False).

key_columns = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "translation": ["product_category_name"]
}

key_report = []

for table, columns in key_columns.items():
    df = data[table]

    key_report.append({
        "table": table,
        "key": " + ".join(columns),
        "missing_key_rows": int(
            df[columns].isna().any(axis=1).sum()
        ),
        "duplicate_key_rows": int(
            df.duplicated(subset=columns, keep=False).sum()
        )
    })

display(pd.DataFrame(key_report))


,table,key,missing_key_rows,duplicate_key_rows
0,customers,customer_id,0,0
1,orders,order_id,0,0
2,items,order_id + order_item_id,0,0
3,payments,order_id + payment_sequential,0,0
4,reviews,review_id + order_id,0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,translation,product_category_name,0,0


In [9]:
# Cell 3.6 — Kiểm tra liên kết giữa các bảng
# Mục đích: Tìm dòng thiếu giá trị liên kết và dòng có mã không tồn tại ở bảng đích.
# Phát hiện: Sáu liên kết chính đều khớp. Products có 610 dòng thiếu danh mục;
# 13 sản phẩm có danh mục không khớp bảng dịch, thuộc 2 giá trị khác nhau.
# Chỉ kiểm tra chiều tham chiếu đã liệt kê; chưa kiểm tra chiều ngược lại.

relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    (
        "products", "product_category_name",
        "translation", "product_category_name"
    ),
]

relationship_report = []

for child_table, child_col, parent_table, parent_col in relationships:
    child = data[child_table][child_col]
    parent = data[parent_table][parent_col].dropna()

    missing = child.isna()
    unmatched = child.notna() & ~child.isin(parent)

    relationship_report.append({
        "relationship": (
            f"{child_table}.{child_col} → "
            f"{parent_table}.{parent_col}"
        ),
        "missing_rows": int(missing.sum()),
        "unmatched_rows": int(unmatched.sum()),
        "unmatched_distinct_values": int(
            child[unmatched].nunique()
        )
    })

display(pd.DataFrame(relationship_report))


,relationship,missing_rows,unmatched_rows,unmatched_distinct_values
0,orders.customer_id → customers.customer_id,0,0,0
1,items.order_id → orders.order_id,0,0,0
2,items.product_id → products.product_id,0,0,0
3,items.seller_id → sellers.seller_id,0,0,0
4,payments.order_id → orders.order_id,0,0,0
5,reviews.order_id → orders.order_id,0,0,0
6,products.product_category_name → translation.p...,610,13,2


In [10]:
# Cell 3.7 — Xem danh mục chưa có bản dịch
# Mục đích: Liệt kê và đếm sản phẩm theo các danh mục không có trong translation.
# Phát hiện: portateis_cozinha_e_preparadores_de_alimentos: 10 sản phẩm; pc_gamer: 3.
# Giữ tên gốc để truy vết; chưa tự thêm bản dịch hoặc loại sản phẩm.

products = data["products"]
translation = data["translation"]

unmatched_mask = (
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(
        translation["product_category_name"].dropna()
    )
)

display(
    products.loc[unmatched_mask, "product_category_name"]
    .value_counts()
    .rename_axis("category")
    .to_frame("product_count")
)


,product_count
category,
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


In [11]:
# Cell 3.8 — Kiểm tra geolocation và mã bưu chính
# Mục đích: Đếm dòng trùng hoàn toàn, mã bưu chính thiếu và mã bưu chính lặp.
# Phát hiện: 1.000.163 dòng; 261.831 dòng trùng hoàn toàn tính dư; 0 dòng thiếu mã bưu chính.
# Có 19.015 mã khác nhau và 999.120 dòng thuộc nhóm có mã lặp.
# Một mã có nhiều tọa độ: mã lặp chưa phải lỗi, cần chú ý khi ghép bảng.

geo = data["geolocation"]
zip_col = "geolocation_zip_code_prefix"

geo_report = pd.DataFrame([{
    "total_rows": len(geo),
    "exact_duplicate_rows": int(geo.duplicated().sum()),
    "missing_zip_rows": int(geo[zip_col].isna().sum()),
    "unique_zip_prefixes": geo[zip_col].nunique(),
    "rows_with_repeated_zip": int(
        geo.duplicated(subset=[zip_col], keep=False).sum()
    )
}])

display(geo_report)


,total_rows,exact_duplicate_rows,missing_zip_rows,unique_zip_prefixes,rows_with_repeated_zip
0,1000163,261831,0,19015,999120


In [12]:
# Cell 3.9 — Kiểm tra tọa độ
# Mục đích: Dùng biến tạm để kiểm tra missing, khả năng chuyển thành số và giới hạn lat/lng.
# Phát hiện: Vĩ độ và kinh độ đều có 0 missing, 0 lỗi chuyển số và 0 giá trị ngoài giới hạn toàn cầu.
# Kiểm tra này chưa xác nhận tọa độ nằm tại Brazil hoặc đúng vị trí thực tế.

geo = data["geolocation"]

lat = pd.to_numeric(geo["geolocation_lat"], errors="coerce")
lng = pd.to_numeric(geo["geolocation_lng"], errors="coerce")

coordinate_report = pd.DataFrame({
    "column": ["geolocation_lat", "geolocation_lng"],
    "missing_rows": [
        geo["geolocation_lat"].isna().sum(),
        geo["geolocation_lng"].isna().sum()
    ],
    "invalid_numeric_rows": [
        (geo["geolocation_lat"].notna() & lat.isna()).sum(),
        (geo["geolocation_lng"].notna() & lng.isna()).sum()
    ],
    "out_of_range_rows": [
        (lat.notna() & ~lat.between(-90, 90)).sum(),
        (lng.notna() & ~lng.between(-180, 180)).sum()
    ]
})

display(coordinate_report)


,column,missing_rows,invalid_numeric_rows,out_of_range_rows
0,geolocation_lat,0,0,0
1,geolocation_lng,0,0,0


In [13]:
# Cell 3.10 — Miền giá trị của giao dịch và điểm đánh giá
# Mục đích: Kiểm tra missing, lỗi chuyển số, miền giá trị và tính nguyên khi cần.
# Phát hiện: Có 2 dòng payment_installments < 1; các kiểm tra còn lại đều ghi nhận 0.
# Giá/phí bằng 0 được chấp nhận trong kiểm tra sơ bộ; không áp dụng ngưỡng trên cho tiền.

rules = [
    ("items", "price", 0, None, False),
    ("items", "freight_value", 0, None, False),
    ("items", "order_item_id", 1, None, True),
    ("payments", "payment_value", 0, None, False),
    ("payments", "payment_sequential", 1, None, True),
    ("payments", "payment_installments", 1, None, True),
    ("reviews", "review_score", 1, 5, True),
]

report = []

for table, column, minimum, maximum, integer_required in rules:
    original = data[table][column]
    values = pd.to_numeric(original, errors="coerce")

    outside_range = values.notna() & (values < minimum)

    if maximum is not None:
        outside_range |= values.notna() & (values > maximum)

    non_integer = (
        values.notna() & (values % 1 != 0)
        if integer_required
        else pd.Series(False, index=values.index)
    )

    report.append({
        "table": table,
        "column": column,
        "missing_rows": int(original.isna().sum()),
        "invalid_numeric_rows": int(
            (original.notna() & values.isna()).sum()
        ),
        "outside_expected_range": int(outside_range.sum()),
        "non_integer_rows": int(non_integer.sum())
    })

display(pd.DataFrame(report))


,table,column,missing_rows,invalid_numeric_rows,outside_expected_range,non_integer_rows
0,items,price,0,0,0,0
1,items,freight_value,0,0,0,0
2,items,order_item_id,0,0,0,0
3,payments,payment_value,0,0,0,0
4,payments,payment_sequential,0,0,0,0
5,payments,payment_installments,0,0,2,0
6,reviews,review_score,0,0,0,0


In [15]:
# Cell 3.11 — Xem thanh toán có số kỳ bất thường
# Mục đích: Xem phương thức, số kỳ và số tiền của các dòng payment_installments < 1.
# Phát hiện: Có 2 dòng credit_card, số kỳ = 0, payment_sequential = 2, số tiền 58.69 và 129.94.
# Dự kiến giữ bản ghi và tiền; số kỳ 0 cần đánh dấu khi làm sạch, chưa tự điền 1.

payments = data["payments"]

installments = pd.to_numeric(
    payments["payment_installments"],
    errors="coerce"
)

suspect_payments = payments.loc[
    installments < 1,
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

display(suspect_payments)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [16]:
# Cell 3.12 — Kiểm tra thứ tự các mốc thời gian
# Mục đích: Chuyển ngày trong bảng tạm và so sánh chỉ khi có đủ hai mốc ngày.
# Phát hiện: 166 đơn bàn giao vận chuyển trước khi mua; 23 đơn giao khách trước khi bàn giao vận chuyển.
# 0 đơn duyệt trước khi mua; 0 đơn giao khách trước khi mua. Các nhóm có thể chồng nhau.
# Ngày thiếu hoặc không chuyển được sẽ không tham gia so sánh; không tự sửa ngày gốc.

orders = data["orders"]

# Chuyển sang datetime trong bảng tạm để kiểm tra
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

dates = orders[date_cols].apply(
    lambda column: pd.to_datetime(
        column,
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )
)

checks = {
    "Duyệt trước khi mua": (
        "order_approved_at",
        "order_purchase_timestamp"
    ),
    "Bàn giao vận chuyển trước khi mua": (
        "order_delivered_carrier_date",
        "order_purchase_timestamp"
    ),
    "Giao khách trước khi bàn giao vận chuyển": (
        "order_delivered_customer_date",
        "order_delivered_carrier_date"
    ),
    "Giao khách trước khi mua": (
        "order_delivered_customer_date",
        "order_purchase_timestamp"
    )
}

report = []

for rule, (later_col, earlier_col) in checks.items():
    comparable = dates[[later_col, earlier_col]].notna().all(axis=1)
    abnormal = comparable & (dates[later_col] < dates[earlier_col])

    report.append({
        "check": rule,
        "comparable_rows": int(comparable.sum()),
        "abnormal_rows": int(abnormal.sum())
    })

display(pd.DataFrame(report))


,check,comparable_rows,abnormal_rows
0,Duyệt trước khi mua,99281,0
1,Bàn giao vận chuyển trước khi mua,97658,166
2,Giao khách trước khi bàn giao vận chuyển,96475,23
3,Giao khách trước khi mua,96476,0


In [17]:
# Cell 3.13 — Miền giá trị của thông số sản phẩm
# Mục đích: Kiểm tra missing, lỗi chuyển số, giá trị âm và bằng 0 ở thông số sản phẩm.
# Phát hiện: Không có giá trị âm hoặc lỗi chuyển số. Có 4 sản phẩm cân nặng = 0.
# Missing giống mục 3.3; số ảnh bằng 0, nếu xuất hiện, cần xét theo ngữ cảnh.

columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

report = []

for column in columns:
    original = data["products"][column]
    values = pd.to_numeric(original, errors="coerce")

    report.append({
        "column": column,
        "missing_rows": int(original.isna().sum()),
        "invalid_numeric_rows": int(
            (original.notna() & values.isna()).sum()
        ),
        "negative_rows": int((values < 0).sum()),
        "zero_rows": int((values == 0).sum())
    })

display(pd.DataFrame(report))


,column,missing_rows,invalid_numeric_rows,negative_rows,zero_rows
0,product_name_lenght,610,0,0,0
1,product_description_lenght,610,0,0,0
2,product_photos_qty,610,0,0,0
3,product_weight_g,2,0,0,4
4,product_length_cm,2,0,0,0
5,product_height_cm,2,0,0,0
6,product_width_cm,2,0,0,0


In [18]:
# Cell 3.14 — Xem sản phẩm có cân nặng bằng 0
# Mục đích: Xem danh mục và kích thước của 4 sản phẩm có cân nặng = 0.
# Phát hiện: Cả 4 thuộc cama_mesa_banho, kích thước dài 30 × cao 25 × rộng 30 cm.
# Cân nặng 0 đáng nghi; chưa có căn cứ suy ra cân nặng đúng từ kích thước.
# Dự kiến giữ sản phẩm và đánh dấu cân nặng khi làm sạch.

products = data["products"]

weight = pd.to_numeric(
    products["product_weight_g"],
    errors="coerce"
)

display(
    products.loc[
        weight.eq(0),
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ]
)


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0,30,25,30
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0,30,25,30
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0,30,25,30
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0,30,25,30


## Kết thúc bước 3

Đã kiểm tra missing, dòng trùng, bộ khóa dự kiến, liên kết, geolocation, miền giá trị và thứ tự thời gian.

- Các bộ khóa dự kiến đều không thiếu và không lặp; reviews dùng cặp `review_id + order_id`.
- Ghi nhận missing trong reviews, orders, products và 13 sản phẩm thuộc 2 danh mục chưa có bản dịch.
- Ghi nhận geolocation trùng hoàn toàn và mã bưu chính có nhiều tọa độ.
- Cần xem xét 2 thanh toán có số kỳ 0, 4 sản phẩm cân nặng 0 và các mốc thời gian bất thường.
- Các số lượng bất thường thuộc các nhóm khác nhau, có thể chồng nhau; không cộng thành số bản ghi lỗi duy nhất.

Quyết định làm sạch mới là dự kiến. Notebook hiện chưa xóa dòng, điền missing, thêm cột phân tích hoặc xuất CSV đã xử lý.


In [19]:
# Cell 4.1 — Tạo bản dữ liệu làm sạch
# Mục đích:
# - Tạo cleaned từ dữ liệu gốc trong data.
# - Bỏ khoảng trắng đầu/cuối ở các cột thông thường.
# - Chuyển chuỗi rỗng hoặc chỉ chứa khoảng trắng thành missing.
# - Giữ nguyên nội dung văn bản review để phục vụ phân tích sau.

cleaned = {
    table: df.copy(deep=True)
    for table, df in data.items()
}

review_text_columns = {
    "review_comment_title",
    "review_comment_message"
}

normalization_report = []

for table, df in cleaned.items():
    trimmed_cells = 0
    new_missing_cells = 0

    for column in df.select_dtypes(include=["string"]).columns:
        if table == "reviews" and column in review_text_columns:
            continue

        original = df[column]
        stripped = original.str.strip()

        trimmed_cells += int(
            original.ne(stripped).fillna(False).sum()
        )

        normalized = stripped.replace("", pd.NA)

        new_missing_cells += int(
            (original.notna() & normalized.isna()).sum()
        )

        df[column] = normalized

    normalization_report.append({
        "table": table,
        "trimmed_cells": trimmed_cells,
        "new_missing_cells": new_missing_cells,
        "rows": len(df)
    })

display(pd.DataFrame(normalization_report))

,table,trimmed_cells,new_missing_cells,rows
0,customers,0,0,99441
1,geolocation,1,0,1000163
2,items,0,0,112650
3,payments,0,0,103886
4,reviews,0,0,99224
5,orders,0,0,99441
6,products,0,0,32951
7,sellers,0,0,3095
8,translation,0,0,71


In [20]:
# Cell 4.2 — Bỏ dòng trùng hoàn toàn trong bản geolocation dùng tra cứu
# Mục đích:
# - Chỉ giữ một lần xuất hiện của mỗi dòng giống nhau ở tất cả các cột.
# - Chưa loại các dòng chỉ trùng mã bưu chính.
# Phát hiện ở bước 3:
# - Có 261.831 dòng trùng hoàn toàn tính dư.

geo_before = cleaned["geolocation"]

geolocation_unique = (
    geo_before
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

print("Số dòng trước:", len(geo_before))
print("Số dòng sau:", len(geolocation_unique))
print("Số dòng đã loại:", len(geo_before) - len(geolocation_unique))
print(
    "Dòng trùng hoàn toàn còn lại:",
    geolocation_unique.duplicated().sum()
)

Số dòng trước: 1000163
Số dòng sau: 738332
Số dòng đã loại: 261831
Dòng trùng hoàn toàn còn lại: 0


In [22]:
# Cell 4.3 — Tạo bảng tra cứu tọa độ theo mã bưu chính
# Mục đích:
# - Chuyển tọa độ sang kiểu số trong bản làm việc.
# - Lấy trung vị vĩ độ và kinh độ theo mã bưu chính.
# - Tạo bảng có mã bưu chính duy nhất để dùng khi ghép bảng.
# Lưu ý:
# - Tọa độ trung vị là vị trí đại diện, có thể không trùng điểm quan sát nào.
# - Tính từ geolocation_unique nên các dòng trùng hoàn toàn chỉ được tính một lần.

geo_work = geolocation_unique.copy()

for column in ["geolocation_lat", "geolocation_lng"]:
    geo_work[column] = pd.to_numeric(
        geo_work[column],
        errors="coerce"
    )

valid_coordinates = (
    geo_work["geolocation_zip_code_prefix"].notna()
    & geo_work["geolocation_lat"].between(-90, 90)
    & geo_work["geolocation_lng"].between(-180, 180)
)

geolocation_lookup = (
    geo_work.loc[valid_coordinates]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median"),
        source_row_count=("geolocation_lat", "size")
    )
)

print("Số dòng không dùng được:", int((~valid_coordinates).sum()))
print("Số mã bưu chính:", len(geolocation_lookup))
print(
    "Mã bưu chính bị lặp:",
    geolocation_lookup["geolocation_zip_code_prefix"]
    .duplicated()
    .sum()
)

display(geolocation_lookup.head())

Số dòng không dùng được: 0
Số mã bưu chính: 19015
Mã bưu chính bị lặp: 0


,geolocation_zip_code_prefix,latitude,longitude,source_row_count
0,01001,-23.549951,-46.634027,11
1,01002,-23.548228,-46.635247,6
2,01003,-23.548977,-46.635313,11
3,01004,-23.54955,-46.634771,14
4,01005,-23.549763,-46.6361,13


In [23]:
# Cell 4.4 — Chuẩn hóa ngày giờ trong orders
# Mục đích:
# - Chuyển 5 cột ngày sang datetime để tính toán thời gian.
# - Báo cáo giá trị có dữ liệu nhưng không chuyển được.
# - Đánh dấu các mốc thời gian bất thường để lọc khi phân tích.
# Phát hiện ở bước 3:
# - 166 đơn bàn giao vận chuyển trước khi mua.
# - 23 đơn giao khách trước khi bàn giao vận chuyển.
# - 8 đơn delivered thiếu ngày giao thực tế.

orders = cleaned["orders"]

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

conversion_report = []

for column in date_columns:
    # Đọc từ data để có thể chạy lại cell và vẫn giữ báo cáo chính xác
    original = data["orders"][column].str.strip().replace("", pd.NA)

    converted = pd.to_datetime(
        original,
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

    conversion_report.append({
        "column": column,
        "invalid_date_rows": int(
            (original.notna() & converted.isna()).sum()
        )
    })

    orders[column] = converted

# Đánh dấu bất thường; chỉ so sánh khi có đủ hai mốc ngày
time_rules = {
    "flag_approval_before_purchase": (
        "order_approved_at",
        "order_purchase_timestamp"
    ),
    "flag_carrier_before_purchase": (
        "order_delivered_carrier_date",
        "order_purchase_timestamp"
    ),
    "flag_delivery_before_carrier": (
        "order_delivered_customer_date",
        "order_delivered_carrier_date"
    ),
    "flag_delivery_before_purchase": (
        "order_delivered_customer_date",
        "order_purchase_timestamp"
    )
}

for flag, (later_column, earlier_column) in time_rules.items():
    has_both_dates = orders[
        [later_column, earlier_column]
    ].notna().all(axis=1)

    orders[flag] = (
        orders[later_column] < orders[earlier_column]
    ).astype("boolean").where(has_both_dates, pd.NA)

orders["flag_delivered_missing_date"] = (
    orders["order_status"].eq("delivered")
    & orders["order_delivered_customer_date"].isna()
)

display(pd.DataFrame(conversion_report))

flag_columns = list(time_rules) + ["flag_delivered_missing_date"]

display(pd.DataFrame({
    "flag": flag_columns,
    "flagged_rows": [
        int(orders[column].sum())
        for column in flag_columns
    ]
}))

,column,invalid_date_rows
0,order_purchase_timestamp,0
1,order_approved_at,0
2,order_delivered_carrier_date,0
3,order_delivered_customer_date,0
4,order_estimated_delivery_date,0


,flag,flagged_rows
0,flag_approval_before_purchase,0
1,flag_carrier_before_purchase,166
2,flag_delivery_before_carrier,23
3,flag_delivery_before_purchase,0
4,flag_delivered_missing_date,8


In [25]:
# Cell 4.5 — Chuẩn hóa số tiền và số kỳ thanh toán
# Mục đích:
# - Chuyển payment_value và payment_installments sang kiểu số.
# - Giữ payment_installments gốc sau khi chuyển kiểu.
# - Tạo payment_installments_valid cho các số kỳ nguyên >= 1.
# Phát hiện:
# - Có 2 dòng credit_card có số kỳ = 0.
# - Giữ các dòng và số tiền; số kỳ không hợp lệ được để missing
#   trong cột payment_installments_valid.

payments = cleaned["payments"]

for column in ["payment_value", "payment_installments"]:
    payments[column] = pd.to_numeric(
        payments[column],
        errors="coerce"
    )

installments = payments["payment_installments"]

valid_installments = (
    installments.notna()
    & installments.ge(1)
    & installments.mod(1).eq(0)
)

payments["flag_invalid_installments"] = (
    installments.notna() & ~valid_installments
)

payments["payment_installments_valid"] = (
    installments.where(valid_installments).astype("Int64")
)

print(
    "Số dòng có số kỳ không hợp lệ:",
    int(payments["flag_invalid_installments"].sum())
)

display(
    payments.loc[
        payments["flag_invalid_installments"],
        [
            "order_id",
            "payment_type",
            "payment_value",
            "payment_installments",
            "payment_installments_valid"
        ]
    ]
)

Số dòng có số kỳ không hợp lệ: 2


,order_id,payment_type,payment_value,payment_installments,payment_installments_valid
46982,744bade1fcf9ff3f31d860ace076d422,credit_card,58.69,0,<NA>
79014,1a57108394169c0b47d8f876acc9ba2d,credit_card,129.94,0,<NA>


In [26]:
# Cell 4.6 — Chuẩn hóa thông số sản phẩm
# Mục đích:
# - Chuyển các thông số sản phẩm sang kiểu số.
# - Đánh dấu cân nặng <= 0 là không hợp lệ.
# - Tạo cột cân nặng hợp lệ để phân tích.
# Phát hiện:
# - Có 4 sản phẩm cân nặng = 0 và 2 sản phẩm thiếu cân nặng.
# - Giữ các sản phẩm; không tự suy đoán cân nặng thay thế.

products = cleaned["products"]

numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in numeric_columns:
    products[column] = pd.to_numeric(
        products[column],
        errors="coerce"
    )

weight = products["product_weight_g"]

products["flag_invalid_weight"] = (
    weight.notna() & weight.le(0)
)

products["product_weight_g_valid"] = weight.where(
    weight.gt(0)
)

print(
    "Sản phẩm có cân nặng không hợp lệ:",
    int(products["flag_invalid_weight"].sum())
)

print(
    "Sản phẩm thiếu cân nặng trong cột hợp lệ:",
    int(products["product_weight_g_valid"].isna().sum())
)

display(
    products.loc[
        products["flag_invalid_weight"],
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_weight_g_valid"
        ]
    ]
)

Sản phẩm có cân nặng không hợp lệ: 4
Sản phẩm thiếu cân nặng trong cột hợp lệ: 6


,product_id,product_category_name,product_weight_g,product_weight_g_valid
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0,<NA>
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0,<NA>
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0,<NA>
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0,<NA>


In [27]:
# Cell 4.7 — Bổ sung bản dịch danh mục sản phẩm
# Mục đích:
# - Ghép bản dịch mà vẫn giữ toàn bộ sản phẩm.
# - Tạo category_for_analysis dùng cho phân tích.
# - Ghi rõ nguồn của nhãn danh mục.
# Phát hiện:
# - 610 sản phẩm thiếu danh mục.
# - 13 sản phẩm thuộc 2 danh mục chưa có bản dịch.

# Bỏ các cột đã tạo để có thể chạy lại cell
products = cleaned["products"].drop(
    columns=[
        "product_category_name_english",
        "category_for_analysis",
        "category_label_source"
    ],
    errors="ignore"
)

rows_before = len(products)

products = products.merge(
    cleaned["translation"][
        ["product_category_name", "product_category_name_english"]
    ],
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

products["category_for_analysis"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
    .fillna("unknown")
)

# Xác định nhãn được lấy từ đâu
products["category_label_source"] = "missing_category"

has_original = products["product_category_name"].notna()
has_translation = products["product_category_name_english"].notna()

products.loc[
    has_original & ~has_translation,
    "category_label_source"
] = "original_name"

products.loc[
    has_translation,
    "category_label_source"
] = "english_translation"

assert len(products) == rows_before, "Số dòng sản phẩm bị thay đổi!"

cleaned["products"] = products

display(
    products["category_label_source"]
    .value_counts()
    .rename_axis("label_source")
    .to_frame("product_count")
)

,product_count
label_source,
english_translation,32328
missing_category,610
original_name,13


In [28]:
# Cell 4.8 — Chuẩn hóa dữ liệu reviews
# Mục đích:
# - Chuyển review_score sang kiểu số.
# - Tạo cột điểm hợp lệ: số nguyên từ 1 đến 5.
# - Đánh dấu có tiêu đề/nội dung bình luận.
# Phát hiện:
# - Điểm đánh giá đã vượt qua kiểm tra miền giá trị.
# - Nhiều review thiếu phần văn bản; vẫn giữ các dòng này.

reviews = cleaned["reviews"]

reviews["review_score"] = pd.to_numeric(
    reviews["review_score"],
    errors="coerce"
)

score = reviews["review_score"]

valid_score = (
    score.notna()
    & score.between(1, 5)
    & score.mod(1).eq(0)
)

reviews["review_score_valid"] = (
    score.where(valid_score).astype("Int64")
)

reviews["flag_invalid_score"] = (
    score.notna() & ~valid_score
)

for column, flag in [
    ("review_comment_title", "has_comment_title"),
    ("review_comment_message", "has_comment_message")
]:
    reviews[flag] = (
        reviews[column]
        .fillna("")
        .str.strip()
        .ne("")
    )

print("Tổng số review:", len(reviews))
print("Điểm không hợp lệ:", int(reviews["flag_invalid_score"].sum()))
print("Điểm thiếu:", int(reviews["review_score_valid"].isna().sum()))
print("Có tiêu đề:", int(reviews["has_comment_title"].sum()))
print("Có nội dung:", int(reviews["has_comment_message"].sum()))

Tổng số review: 99224
Điểm không hợp lệ: 0
Điểm thiếu: 0
Có tiêu đề: 11566
Có nội dung: 40950


In [29]:
# Cell 4.9 — Chuẩn hóa giá và phí vận chuyển
# Mục đích:
# - Chuyển price và freight_value sang kiểu số.
# - Kiểm tra lỗi chuyển đổi và tổng tiền trước/sau.
# Phát hiện ở bước 3:
# - Hai cột không thiếu, không có số âm hoặc lỗi chuyển số.

items = cleaned["items"]

conversion_report = []

for column in ["price", "freight_value"]:
    source = pd.to_numeric(
        data["items"][column],
        errors="coerce"
    )

    original = items[column]
    converted = pd.to_numeric(original, errors="coerce")

    conversion_report.append({
        "column": column,
        "invalid_numeric_rows": int(
            (original.notna() & converted.isna()).sum()
        ),
        "missing_rows": int(converted.isna().sum()),
        "negative_rows": int(converted.lt(0).sum()),
        "total_before": source.sum(min_count=1),
        "total_after": converted.sum(min_count=1)
    })

    items[column] = converted

display(pd.DataFrame(conversion_report).round(2))

,column,invalid_numeric_rows,missing_rows,negative_rows,total_before,total_after
0,price,0,0,0,13591643.70,13591643.70
1,freight_value,0,0,0,2251909.54,2251909.54


In [30]:
# Cell 4.10 — Chuẩn hóa các cột ngày còn lại
# Mục đích:
# - Chuyển hạn bàn giao hàng và thời gian review sang datetime.
# - Báo cáo ngày thiếu và giá trị không chuyển đổi được.
# - Giữ nguyên số dòng.

date_fields = {
    "items": ["shipping_limit_date"],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

date_report = []

for table, columns in date_fields.items():
    for column in columns:
        # Dùng nguồn gốc để báo cáo nhất quán khi chạy lại cell
        original = (
            data[table][column]
            .str.strip()
            .replace("", pd.NA)
        )

        converted = pd.to_datetime(
            original,
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        cleaned[table][column] = converted

        date_report.append({
            "table": table,
            "column": column,
            "missing_before": int(original.isna().sum()),
            "invalid_date_rows": int(
                (original.notna() & converted.isna()).sum()
            ),
            "missing_after": int(converted.isna().sum())
        })

display(pd.DataFrame(date_report))

,table,column,missing_before,invalid_date_rows,missing_after
0,items,shipping_limit_date,0,0,0
1,reviews,review_creation_date,0,0,0
2,reviews,review_answer_timestamp,0,0,0


In [31]:
# Cell 4.11 — Kiểm tra số dòng và khóa sau làm sạch
# Mục đích:
# - Xác nhận các bảng vẫn giữ đủ bản ghi.
# - Kiểm tra khóa không bị thiếu hoặc lặp sau các bước xử lý.
# - Kiểm tra bảng tọa độ tra cứu có một dòng cho mỗi mã bưu chính.

key_columns = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "translation": ["product_category_name"]
}

validation_report = []

for table, columns in key_columns.items():
    df = cleaned[table]

    validation_report.append({
        "table": table,
        "rows_before": len(data[table]),
        "rows_after": len(df),
        "missing_key_rows": int(
            df[columns].isna().any(axis=1).sum()
        ),
        "duplicate_key_rows": int(
            df.duplicated(columns, keep=False).sum()
        )
    })

validation_report = pd.DataFrame(validation_report)
display(validation_report)

assert (
    validation_report["rows_before"]
    == validation_report["rows_after"]
).all(), "Có bảng bị thay đổi số dòng!"

assert validation_report["missing_key_rows"].eq(0).all(), (
    "Có bảng bị thiếu khóa!"
)

assert validation_report["duplicate_key_rows"].eq(0).all(), (
    "Có bảng bị lặp khóa!"
)

zip_column = "geolocation_zip_code_prefix"

assert geolocation_lookup[zip_column].notna().all(), (
    "Bảng tra cứu thiếu mã bưu chính!"
)

assert geolocation_lookup[zip_column].is_unique, (
    "Bảng tra cứu bị lặp mã bưu chính!"
)

assert len(geolocation_lookup) == (
    geolocation_unique[zip_column].dropna().nunique()
), "Có mã bưu chính chưa có tọa độ đại diện!"

print("Đạt kiểm tra số dòng và khóa.")
print("Số mã trong bảng tra cứu tọa độ:", len(geolocation_lookup))

,table,rows_before,rows_after,missing_key_rows,duplicate_key_rows
0,customers,99441,99441,0,0
1,orders,99441,99441,0,0
2,items,112650,112650,0,0
3,payments,103886,103886,0,0
4,reviews,99224,99224,0,0
5,products,32951,32951,0,0
6,sellers,3095,3095,0,0
7,translation,71,71,0,0


Đạt kiểm tra số dòng và khóa.
Số mã trong bảng tra cứu tọa độ: 19015


In [32]:
# Cell 4.12 — Xuất dữ liệu đã làm sạch
# Mục đích:
# - Lưu 8 bảng đã xử lý vào data/processed.
# - Lưu riêng geolocation đã bỏ dòng trùng và bảng tọa độ đại diện.
# - Báo cáo tên file, số dòng và số cột đã xuất.

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_tables = {
    f"{table}_clean": df
    for table, df in cleaned.items()
    if table != "geolocation"
}

export_tables["geolocation_unique"] = geolocation_unique
export_tables["geolocation_lookup"] = geolocation_lookup

export_report = []

for name, df in export_tables.items():
    file_path = OUTPUT_DIR / f"{name}.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    export_report.append({
        "file": file_path.name,
        "rows": len(df),
        "columns": len(df.columns)
    })

display(pd.DataFrame(export_report))
print("Đã lưu tại:", OUTPUT_DIR.resolve())

,file,rows,columns
0,customers_clean.csv,99441,5
1,items_clean.csv,112650,7
2,payments_clean.csv,103886,7
3,reviews_clean.csv,99224,11
4,orders_clean.csv,99441,13
5,products_clean.csv,32951,14
6,sellers_clean.csv,3095,4
7,translation_clean.csv,71,2
8,geolocation_unique.csv,738332,5
9,geolocation_lookup.csv,19015,4


Đã lưu tại: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11\data\processed


In [33]:
# Cell 5.1 — Tổng hợp items về cấp đơn hàng
# Mục đích:
# - Tạo một dòng cho mỗi order_id có mặt hàng.
# - Tính số mặt hàng, số sản phẩm/người bán khác nhau.
# - Tính tổng giá hàng và phí vận chuyển.
# Lưu ý:
# - item_count đếm dòng mặt hàng.
# - items_value là tổng giá hàng, chưa cộng phí vận chuyển.

items = cleaned["items"]

order_items_agg = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        distinct_product_count=("product_id", "nunique"),
        distinct_seller_count=("seller_id", "nunique"),
        items_value=("price", lambda x: x.sum(min_count=1)),
        freight_value=("freight_value", lambda x: x.sum(min_count=1))
    )
)

order_items_agg["items_plus_freight"] = (
    order_items_agg["items_value"]
    + order_items_agg["freight_value"]
)

# Kiểm tra tổng hợp có bảo toàn số mặt hàng và tổng tiền
assert order_items_agg["order_id"].is_unique
assert order_items_agg["item_count"].sum() == len(items)

for source_column, result_column in [
    ("price", "items_value"),
    ("freight_value", "freight_value")
]:
    before = items[source_column].sum()
    after = order_items_agg[result_column].sum()

    assert abs(before - after) < 0.01, (
        f"Tổng {source_column} bị thay đổi!"
    )

print("Số đơn có mặt hàng:", len(order_items_agg))
display(order_items_agg.head())

Số đơn có mặt hàng: 98666


,order_id,item_count,distinct_product_count,distinct_seller_count,items_value,freight_value,items_plus_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.9,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.9,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.0,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.9,18.14,218.04


In [34]:
# Cell 5.2 — Tổng hợp payments về cấp đơn hàng
# Mục đích:
# - Tính tổng tiền thanh toán của mỗi đơn.
# - Đếm số bản ghi và số phương thức thanh toán khác nhau.
# - Giữ danh sách phương thức để tra cứu.
# Lưu ý:
# - Không cộng số kỳ trả góp giữa các bản ghi thanh toán.

payments = cleaned["payments"]

payments_agg = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value_total=(
            "payment_value",
            lambda x: x.sum(min_count=1)
        ),
        payment_record_count=("payment_sequential", "size"),
        payment_type_count=("payment_type", "nunique"),
        payment_types=(
            "payment_type",
            lambda x: "|".join(sorted(x.dropna().unique()))
        ),
        invalid_installments_record_count=(
            "flag_invalid_installments", "sum"
        )
    )
)

# Kiểm tra số bản ghi và tổng tiền được bảo toàn
assert payments_agg["order_id"].is_unique

assert (
    payments_agg["payment_record_count"].sum()
    == len(payments)
)

assert abs(
    payments_agg["payment_value_total"].sum()
    - payments["payment_value"].sum()
) < 0.01, "Tổng tiền thanh toán bị thay đổi!"

print("Số đơn có thanh toán:", len(payments_agg))
display(payments_agg.head())

Số đơn có thanh toán: 99440


,order_id,payment_value_total,payment_record_count,payment_type_count,payment_types,invalid_installments_record_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,credit_card,0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,credit_card,0
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,credit_card,0
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,credit_card,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,credit_card,0


In [35]:
# Cell 5.3 — Tổng hợp reviews về cấp đơn hàng
# Mục đích:
# - Tạo một dòng cho mỗi order_id có review.
# - Giữ số review và số review có điểm hợp lệ.
# - Tính điểm trung bình, thấp nhất và cao nhất.
# Lưu ý:
# - Điểm trung bình là cách tổng hợp được chọn cho bảng cấp đơn hàng.
# - Bảng reviews chi tiết vẫn dùng để xem từng đánh giá.

reviews = cleaned["reviews"]

reviews_agg = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_count=("review_id", "size"),
        valid_score_count=("review_score_valid", "count"),
        mean_review_score=("review_score_valid", "mean"),
        min_review_score=("review_score_valid", "min"),
        max_review_score=("review_score_valid", "max"),
        reviews_with_message=("has_comment_message", "sum")
    )
)

# Kiểm tra số review được bảo toàn
assert reviews_agg["order_id"].is_unique

assert reviews_agg["review_count"].sum() == len(reviews)

assert (
    reviews_agg["valid_score_count"].sum()
    == reviews["review_score_valid"].notna().sum()
)

print("Số đơn có review:", len(reviews_agg))
print(
    "Số đơn có nhiều hơn một review:",
    int(reviews_agg["review_count"].gt(1).sum())
)

display(reviews_agg.head())

Số đơn có review: 98673
Số đơn có nhiều hơn một review: 547


,order_id,review_count,valid_score_count,mean_review_score,min_review_score,max_review_score,reviews_with_message
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,5.0,5,5,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,4.0,4,4,0
2,000229ec398224ef6ca0657da4fc703e,1,1,5.0,5,5,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,4.0,4,4,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,5.0,5,5,1


In [36]:
# Cell 5.4 — Tạo bảng phân tích cấp đơn hàng
# Mục đích:
# - Ghép orders với thông tin khách hàng và các bảng tổng hợp.
# - Giữ toàn bộ đơn hàng bằng left join.
# - Kiểm soát quan hệ ghép để tránh nhân số dòng.
# - Đánh dấu đơn có dữ liệu items, payments và reviews.

orders_analysis = (
    cleaned["orders"]
    .merge(
        cleaned["customers"],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        order_items_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payments_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        reviews_agg,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

orders_analysis["has_items"] = (
    orders_analysis["item_count"].notna()
)

orders_analysis["has_payments"] = (
    orders_analysis["payment_record_count"].notna()
)

orders_analysis["has_reviews"] = (
    orders_analysis["review_count"].notna()
)

# Kiểm tra số đơn không bị thay đổi sau khi ghép
assert len(orders_analysis) == len(cleaned["orders"])
assert orders_analysis["order_id"].is_unique
assert orders_analysis["order_id"].notna().all()

print("Số đơn sau khi ghép:", len(orders_analysis))

coverage_report = pd.DataFrame([
    {
        "data_source": source,
        "orders_with_data": int(orders_analysis[flag].sum()),
        "orders_without_data": int((~orders_analysis[flag]).sum())
    }
    for source, flag in [
        ("items", "has_items"),
        ("payments", "has_payments"),
        ("reviews", "has_reviews")
    ]
])

display(coverage_report)
display(orders_analysis.head())

Số đơn sau khi ghép: 99441


,data_source,orders_with_data,orders_without_data
0,items,98666,775
1,payments,99440,1
2,reviews,98673,768


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,flag_approval_before_purchase,flag_carrier_before_purchase,...,invalid_installments_record_count,review_count,valid_score_count,mean_review_score,min_review_score,max_review_score,reviews_with_message,has_items,has_payments,has_reviews
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,...,0,1.0,1,4.0,4,4,1,True,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,...,0,1.0,1,4.0,4,4,1,True,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,...,0,1.0,1,5.0,5,5,0,True,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,...,0,1.0,1,5.0,5,5,1,True,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,...,0,1.0,1,5.0,5,5,0,True,True,True


In [37]:
# Cell 5.5 — Tạo biến phân tích giao hàng
# Mục đích:
# - Tính số ngày từ lúc mua đến khi giao khách.
# - Tính giao trễ theo ngày lịch so với ngày dự kiến.
# - Giữ missing nếu chưa đủ thông tin để xác định.
# Lưu ý:
# - Chỉ tính cho đơn delivered có ngày mua/giao hợp lệ.
# - Các cờ bất thường ở công đoạn khác vẫn được giữ để lọc thêm khi cần.

purchase = orders_analysis["order_purchase_timestamp"]
actual = orders_analysis["order_delivered_customer_date"]
estimated = orders_analysis["order_estimated_delivery_date"]

valid_delivery = (
    orders_analysis["order_status"].eq("delivered")
    & purchase.notna()
    & actual.notna()
    & actual.ge(purchase)
)

# Thời gian thực tế, có phần thập phân
orders_analysis["delivery_days"] = (
    (actual - purchase).dt.total_seconds() / 86400
).where(valid_delivery)

# Giao trễ theo ngày lịch
valid_delay = valid_delivery & estimated.notna()

orders_analysis["delay_days"] = (
    actual.dt.normalize() - estimated.dt.normalize()
).dt.days.where(valid_delay).astype("Int64")

orders_analysis["is_late"] = (
    orders_analysis["delay_days"]
    .gt(0)
    .astype("boolean")
    .where(valid_delay, pd.NA)
)

print(
    "Số đơn tính được thời gian giao:",
    orders_analysis["delivery_days"].notna().sum()
)

display(
    orders_analysis["is_late"]
    .value_counts(dropna=False)
    .rename_axis("is_late")
    .to_frame("order_count")
)

Số đơn tính được thời gian giao: 96470


,order_count
is_late,
False,89936
True,6534
<NA>,2971


In [38]:
# Cell 5.6 — Đối chiếu tiền ở cấp đơn hàng
# Mục đích:
# - So sánh tổng thanh toán với tổng giá hàng + phí vận chuyển.
# - Đánh dấu chênh lệch lớn hơn 0.01.
# - Giữ missing nếu đơn thiếu một trong hai tổng tiền.
# Lưu ý:
# - Chênh lệch cần xem thêm; chưa tự kết luận là lỗi.
# - Không sửa tiền để ép hai tổng bằng nhau.

comparable = (
    orders_analysis["payment_value_total"].notna()
    & orders_analysis["items_plus_freight"].notna()
)

orders_analysis["payment_difference"] = (
    orders_analysis["payment_value_total"]
    - orders_analysis["items_plus_freight"]
).round(2).where(comparable)

orders_analysis["flag_payment_difference"] = (
    orders_analysis["payment_difference"]
    .abs()
    .gt(0.01)
    .astype("boolean")
    .where(comparable, pd.NA)
)

print("Số đơn đủ dữ liệu đối chiếu:", int(comparable.sum()))
print(
    "Số đơn chênh lệch lớn hơn 0.01:",
    int(orders_analysis["flag_payment_difference"].sum())
)

display(
    orders_analysis.loc[
        orders_analysis["flag_payment_difference"].fillna(False),
        [
            "order_id",
            "order_status",
            "items_plus_freight",
            "payment_value_total",
            "payment_difference"
        ]
    ].head(10)
)

Số đơn đủ dữ liệu đối chiếu: 98665
Số đơn chênh lệch lớn hơn 0.01: 303


,order_id,order_status,items_plus_freight,payment_value_total,payment_difference
464,8adafb3466daa5395694d3a906ff9d40,delivered,218.02,218.0,-0.02
867,bb2e64c3040ceb9b7ca2bfc602adca08,delivered,257.36,257.34,-0.02
1080,84d6d9710c8af32b5e88f2d1c14ab871,delivered,57.68,61.7,4.02
1126,74016effecaa79d592487f6a4ee47d4b,delivered,51.51,56.96,5.45
1669,239f380355f65dcb68551f07d16fc4a8,delivered,222.63,251.63,29.0
1729,4c57f545143e8865ca2347d8cba154a7,delivered,139.61,151.01,11.4
1986,6e57e23ecac1ae881286657694444267,delivered,350.41,333.91,-16.5
2277,b38b3526b8b8fdc807e8a0a42ab78573,delivered,30.06,30.19,0.13
2357,051fcda88d997d3ff86012da2a556342,delivered,56.6,51.7,-4.9
2541,8b5058499c412c6cf8d013de40e4f9d2,delivered,48.77,51.02,2.25


In [39]:
# Cell 5.7 — Kiểm tra và lưu các bảng phân tích
# Mục đích:
# - Xác nhận bảng phân tích vẫn có một dòng cho mỗi đơn hàng.
# - Kiểm tra thời gian giao và điểm đánh giá sau khi tổng hợp.
# - Lưu kết quả vào data/processed.

assert len(orders_analysis) == len(cleaned["orders"])
assert orders_analysis["order_id"].notna().all()
assert orders_analysis["order_id"].is_unique

assert (
    orders_analysis["delivery_days"]
    .dropna()
    .ge(0)
    .all()
), "Có thời gian giao hàng âm!"

assert (
    orders_analysis["mean_review_score"]
    .dropna()
    .between(1, 5)
    .all()
), "Có điểm trung bình ngoài khoảng 1–5!"

# Đơn chưa xác định được độ trễ phải giữ is_late bị thiếu
unknown_delay = orders_analysis["delay_days"].isna()

assert orders_analysis.loc[
    unknown_delay, "is_late"
].isna().all(), "Có đơn thiếu độ trễ nhưng lại được gán nhãn giao trễ!"

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_tables = {
    "orders_analysis": orders_analysis,
    "order_items_agg": order_items_agg,
    "payments_agg": payments_agg,
    "reviews_agg": reviews_agg
}

saved_report = []

for name, df in analysis_tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")

    saved_report.append({
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns)
    })

display(pd.DataFrame(saved_report))
print("Đã lưu tại:", OUTPUT_DIR.resolve())

,file,rows,columns
0,orders_analysis.csv,99441,42
1,order_items_agg.csv,98666,7
2,payments_agg.csv,99440,6
3,reviews_agg.csv,98673,7


Đã lưu tại: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11\data\processed


In [40]:
# Cell 5.8 — Tạo bảng phân tích cấp mặt hàng
# Mục đích:
# - Phân tích giá trị hàng bán theo sản phẩm, danh mục và người bán.
# - Giữ một dòng cho mỗi order_id + order_item_id.
# - Kiểm tra số dòng và tổng tiền sau khi ghép.
# Lưu ý:
# - Chỉ lấy thông tin cần thiết từ bảng đơn hàng.
# - Không đưa tổng tiền cấp đơn hàng vào từng dòng mặt hàng.

order_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "order_status",
    "order_purchase_timestamp"
]

seller_columns = [
    "seller_id",
    "seller_city",
    "seller_state"
]

order_items_analysis = (
    cleaned["items"]
    .merge(
        cleaned["products"],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        cleaned["sellers"][seller_columns],
        on="seller_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        orders_analysis[order_columns],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
)

assert len(order_items_analysis) == len(cleaned["items"])

assert not order_items_analysis.duplicated(
    ["order_id", "order_item_id"]
).any()

for column in ["price", "freight_value"]:
    assert abs(
        order_items_analysis[column].sum()
        - cleaned["items"][column].sum()
    ) < 0.01, f"Tổng {column} bị thay đổi sau khi ghép!"

output_path = Path("data/processed/order_items_analysis.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

order_items_analysis.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Số dòng mặt hàng:", len(order_items_analysis))
print("Đã lưu tại:", output_path.resolve())

display(order_items_analysis.head())

Số dòng mặt hàng: 112650
Đã lưu tại: C:\Users\LENOVO\Downloads\Group5_STAT3013.R11\data\processed\order_items_analysis.csv


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,...,category_for_analysis,category_label_source,seller_city,seller_state,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,cool_stuff,58,598,...,cool_stuff,english_translation,volta redonda,SP,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,delivered,2017-09-13 08:59:02
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,pet_shop,56,239,...,pet_shop,english_translation,sao paulo,SP,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,delivered,2017-04-26 10:53:06
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,moveis_decoracao,59,695,...,furniture_decor,english_translation,borda da mata,MG,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,delivered,2018-01-14 14:33:31
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42,480,...,perfumery,english_translation,franca,SP,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,delivered,2018-08-08 10:00:35
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.9,18.14,ferramentas_jardim,59,409,...,garden_tools,english_translation,loanda,PR,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,delivered,2017-02-04 13:57:51
